# 12_GFS_Forecast_Extraction.ipynb

Downloads NOAA GFS forecast data for a user-specified latitude/longitude and
creates `gfs_features.csv` for the next 72 hours.

> Note: This notebook uses NOAA's NOMADS OpenDAP endpoint. It is intended as
a starting point for your real-time prediction pipeline.


In [ ]:
!pip -q install siphon pandas xarray netCDF4

import pandas as pd
import xarray as xr

lat=float(input("Latitude: "))
lon=float(input("Longitude: "))

# Example NOMADS latest GFS OpenDAP URL.
# Update the date/run if needed.
url="https://nomads.ncep.noaa.gov/dods/gfs_0p25/gfs_latest"

print("Opening:",url)
ds=xr.open_dataset(url)

# Select nearest grid point
point=ds.sel(lat=lat,lon=lon,method="nearest")

# First 72 forecast hours
subset=point.isel(time=slice(0,73))

rows=[]

for i in range(len(subset.time)):
    rows.append({
        "forecast_time":str(subset.time.values[i]),
        "temperature_k":float(subset["tmp2m"].values[i]) if "tmp2m" in subset else None,
        "pressure_pa":float(subset["pressfc"].values[i]) if "pressfc" in subset else None,
        "u_wind":float(subset["ugrd10m"].values[i]) if "ugrd10m" in subset else None,
        "v_wind":float(subset["vgrd10m"].values[i]) if "vgrd10m" in subset else None,
        "relative_humidity":float(subset["rh2m"].values[i]) if "rh2m" in subset else None,
        "precipitation":float(subset["apcpsfc"].values[i]) if "apcpsfc" in subset else None
    })

gfs=pd.DataFrame(rows)

if "temperature_k" in gfs.columns:
    gfs["temperature_c"]=gfs["temperature_k"]-273.15

gfs.to_csv("gfs_features.csv",index=False)

print(gfs.head())
print("\nSaved: gfs_features.csv")

from google.colab import files
files.download("gfs_features.csv")
